# Lab 7 — Agentic RAG on Union Budget 2026-27

## Setup

In [1]:
!pip install -q langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb "langchain-docling[local]" langsmith langchain-groq


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_docling import DoclingLoader

print("All required libraries imported successfully!")

All required libraries imported successfully!


## Q1 — Ingestion


In [4]:
pdf_path = "/content/budget_speech.pdf"

loader = DoclingLoader(
    file_path=pdf_path
)

docs = loader.load()

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-10-29 18:18:45,344 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-10-29 18:18:45,509 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-10-29 18:18:47,035 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-10-29 18:18:47,140 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-10-29 18:18:47,275 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in ava

In [5]:
print("Number of documents:", len(docs))

Number of documents: 171


In [6]:
print(docs[0])

page_content='SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026' metadata={'source': '/content/budget_speech.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 224.4, 't': 164.22384, 'r': 332.07948995799995, 'b': 151.79327999999998, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 16]}]}], 'headings': ['SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 5207691758262864124, 'filename': 'budget_speech.pdf'}}}


In [7]:
for i in range(10):
    print("DOCUMENT:", i)
    print(docs[i].page_content)
    print("METADATA:", docs[i].metadata)
    print("-" * 50)

DOCUMENT: 0
SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026
METADATA: {'source': '/content/budget_speech.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 224.4, 't': 164.22384, 'r': 332.07948995799995, 'b': 151.79327999999998, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 16]}]}], 'headings': ['SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 5207691758262864124, 'filename': 'budget_speech.pdf'}}}
--------------------------------------------------
DOCUMENT: 1
CONTENTS
| - |
| Yuva Shakti and 3 kartavya, Page No. 1.PART - A = 2. Reform Express, Page No. 1.PART - A = 3. First kartavya: to accelerate and sustain economic growth, Page No. 1.PART - A = 3. Second kartavya: fulfil aspirations and build cap

In [8]:
import re

for doc in docs:
    doc.page_content = re.sub(
        r'(?m)^\s*\d+\s*$',
        '',
        doc.page_content
    )

In [9]:
for i in range(4, 9):
    print("DOCUMENT:", i)
    print(docs[i].page_content)
    print("-" * 50)

DOCUMENT: 4
Introduction
1. Since we assumed office 12 years ago, India's economic trajectory has been marked by stability, fiscal discipline, sustained growth and moderate inflation. This is the result of conscious choices we have made, even in times of heightened uncertainty and disruption. Our Government, led by Hon'ble Prime Minister Modi, has decisively and consistently chosen action over ambivalence, reform over rhetoric and people over populism. 2. We have pursued far reaching structural reforms, fiscal prudence and monetary maintaining a strong thrust on public
- stability whilst investment. Keeping atmanirbharta as a lodestar, we have built domestic manufacturing capacity, energy security and reduced critical import dependencies. Simultaneously, we have ensured that citizens benefit from every action of the Government, undertaking reforms to support

employment generation, agricultural productivity, household purchasing
--------------------------------------------------
DOCUME

In [10]:
print(docs[4].metadata)

{'source': '/content/budget_speech.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/10', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 5, 'bbox': {'l': 108.0, 't': 359.904, 'r': 462.51708240000005, 'b': 208.91200000000003, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 550]}]}, {'self_ref': '#/texts/11', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 5, 'bbox': {'l': 108.0, 't': 233.904, 'r': 462.46909199999976, 'b': 148.91200000000003, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 292]}]}, {'self_ref': '#/texts/12', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 6, 'bbox': {'l': 319.92, 't': 751.584, 'r': 326.0027832, 'b': 741.472, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 1]}]}, {'self_ref': '#/texts/13'

In [11]:
for doc in docs:
    page_numbers = []

    for item in doc.metadata["dl_meta"]["doc_items"]:
        for prov in item.get("prov", []):
            page_no = prov.get("page_no")

            if page_no not in page_numbers:
                page_numbers.append(page_no)

    doc.metadata["page_numbers"] = page_numbers

In [12]:
for i in range(5):
    print("DOCUMENT:", i)
    print("PAGE NUMBERS:", docs[i].metadata["page_numbers"])
    print("-" * 40)

DOCUMENT: 0
PAGE NUMBERS: [1]
----------------------------------------
DOCUMENT: 1
PAGE NUMBERS: [3]
----------------------------------------
DOCUMENT: 2
PAGE NUMBERS: [3]
----------------------------------------
DOCUMENT: 3
PAGE NUMBERS: [5]
----------------------------------------
DOCUMENT: 4
PAGE NUMBERS: [5, 6]
----------------------------------------


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [14]:
chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 212


In [15]:
print("CHUNK TEXT:")
print(chunks[0].page_content)

print("\nPAGE NUMBERS:")
print(chunks[0].metadata["page_numbers"])

CHUNK TEXT:
SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026

PAGE NUMBERS:
[1]


## Q2 — Vector Store


In [16]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
for chunk in chunks:
    chunk.metadata = {
        "source": chunk.metadata["source"],
        "page_numbers": chunk.metadata["page_numbers"]
    }

In [18]:
print(chunks[0].metadata)

{'source': '/content/budget_speech.pdf', 'page_numbers': [1]}


In [29]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="/content/chroma_db_clean"
)

print("Chunks stored in ChromaDB:", vectorstore._collection.count())

Chunks stored in ChromaDB: 636


In [30]:
query = "Biopharma SHAKTI outlay"

results = vectorstore.similarity_search_with_score(
    query,
    k=3
)

for i, (doc, score) in enumerate(results, start=1):
    print(f"RESULT: {i}")
    print("PAGE:", doc.metadata["page_numbers"])
    print("SCORE:", score)
    print("TEXT:", doc.page_content)
    print("-" * 50)

RESULT: 1
PAGE: [8]
SCORE: 0.5278290510177612
TEXT: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet global standards and approval timeframes through a dedicated scientif

## Q3 — Create a Budget Retrieval Tool

In [31]:
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq

print("Q3 libraries imported successfully.")

Q3 libraries imported successfully.


In [32]:
@tool
def search_budget(query: str) -> str:
    """Search the Union Budget document for information relevant to the user's question."""

    results = vectorstore.similarity_search_with_score(
        query,
        k=3
    )

    retrieved_context = []

    for doc, score in results:
        pages = doc.metadata.get("page_numbers", ["Unknown"])

        retrieved_context.append(
            f"Page {pages}: {doc.page_content}"
        )

    return "\n\n".join(retrieved_context)

In [33]:
test_result = search_budget.invoke(
    {"query": "What is Biopharma SHAKTI?"}
)

print(test_result)

Page [8]: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet global standards and approval timeframes through a dedicated scientific review cadre and specialists. 17. India

In [43]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an assistant for answering questions about the Union Budget document.

For questions asking for factual information, you MUST use the search_budget tool and base your answer only on the retrieved document content.

Always cite the page number when answering from the document.

If the retrieved document does not contain enough information to answer the question, say:
"The answer is not available in the provided Budget document."

Do not answer factual questions using your own knowledge.

For normal conversational messages such as greetings, you may respond directly without using the tool."""
    ),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{question}")
])

In [44]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


In [50]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

tools = [search_budget]

llm_with_tools = llm.bind_tools(tools)

print("ChatGroq initialized with Budget search tool.")

ChatGroq initialized with Budget search tool.


In [49]:

chat_history = []

question = "What does the Union Budget say about Biopharma SHAKTI?"

messages = prompt.invoke({
    "chat_history": chat_history,
    "question": question
})

response = llm_with_tools.invoke(messages)

print(response)

content='' additional_kwargs={'reasoning_content': 'We need to search the budget document for "Biopharma SHAKTI". Use search_budget.', 'tool_calls': [{'id': 'fc_e2189e74-d865-4036-bfa0-18fb0f05ff6a', 'function': {'arguments': '{"query":"Biopharma SHAKTI"}', 'name': 'search_budget'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 246, 'total_tokens': 295, 'completion_time': 0.060914951, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.011992003, 'prompt_tokens_details': None, 'queue_time': 0.172999623, 'total_time': 0.072906954}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a18911-9201-75d1-9997-5153cde1db21-0' tool_calls=[{'name': 'search_budget', 'args': {'query': 'Biopharma SHAKTI'}, 'id': 'fc_e2189e74-d865-4036-bfa0-18fb0f05ff6a', 'type': 'tool_call'}] invalid_too

In [51]:
print("Tool calls:")
print(response.tool_calls)

Tool calls:
[{'name': 'search_budget', 'args': {'query': 'Biopharma SHAKTI'}, 'id': 'fc_e2189e74-d865-4036-bfa0-18fb0f05ff6a', 'type': 'tool_call'}]


In [52]:
tool_call = response.tool_calls[0]

tool_result = search_budget.invoke(tool_call["args"])

print("TOOL RESULT:")
print(tool_result)

TOOL RESULT:
Page [8]: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet global standards and approval timeframes through a dedicated scientific review cadre and specialis

In [53]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

final_messages = list(messages.messages) + [
    response,
    tool_message
]

final_response = llm_with_tools.invoke(final_messages)

print("FINAL ANSWER:")
print(final_response.content)

FINAL ANSWER:
The Union Budget outlines the **Biopharma SHAKTI** (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) as follows:

- It recognizes that India’s disease burden is shifting toward non‑communicable diseases such as diabetes, cancer, and autoimmune disorders, and stresses the importance of biologic medicines for longevity and quality of life at affordable costs.  
- To position India as a global biopharma manufacturing hub, the Budget proposes **Biopharma SHAKTI** with an outlay of **₹10,000 crores over the next five years**.  
- The strategy will build an ecosystem for domestic production of biologics and biosimilars, including:
  - A biopharma‑focused network with **3 new National Institutes of Pharmaceutical Education and Research (NIPER)** and upgrading **7 existing ones**.  
  - Creation of a network of **over 1,000 accredited India Clinical Trials sites**.  
  - Strengthening the Central Drugs Standard Control Organisation to meet global 

In [55]:
chat_history = []

question = "Hello! How are you?"

messages = prompt.invoke({
    "chat_history": chat_history,
    "question": question
})

response_no_search = llm_with_tools.invoke(messages)

print("ANSWER:")
print(response_no_search.content)

print("\nTOOL CALLS:")
print(response_no_search.tool_calls)


ANSWER:
Hello! I'm doing great, thank you. How can I help you today?

TOOL CALLS:
[]


In [56]:
chat_history = []

question = "Who won the FIFA World Cup in 2018?"

messages = prompt.invoke({
    "chat_history": chat_history,
    "question": question
})

response_missing = llm_with_tools.invoke(messages)

print("INITIAL RESPONSE:")
print(response_missing.content)

print("\nTOOL CALLS:")
print(response_missing.tool_calls)

INITIAL RESPONSE:


TOOL CALLS:
[{'name': 'search_budget', 'args': {'query': 'FIFA World Cup 2018'}, 'id': 'fc_91378b39-c1e6-4f03-a613-69e925006930', 'type': 'tool_call'}]


In [57]:
missing_tool_call = response_missing.tool_calls[0]

missing_tool_result = search_budget.invoke(
    missing_tool_call["args"]
)

print("SEARCH RESULT:")
print(missing_tool_result)

SEARCH RESULT:
Page [19]: Sports

Page [19]: Sports

Page [19]: Sports


In [58]:
from langchain_core.messages import ToolMessage

missing_tool_message = ToolMessage(
    content=missing_tool_result,
    tool_call_id=missing_tool_call["id"]
)

final_missing_messages = list(messages.messages) + [
    response_missing,
    missing_tool_message
]

missing_final_response = llm_with_tools.invoke(
    final_missing_messages
)

print("FINAL ANSWER:")
print(missing_final_response.content)

FINAL ANSWER:



In [59]:
print("FINAL CONTENT:")
print(missing_final_response.content)

print("\nTOOL CALLS:")
print(missing_final_response.tool_calls)

FINAL CONTENT:


TOOL CALLS:
[{'name': 'search_budget', 'args': {'query': 'Page 19'}, 'id': 'fc_259d9a95-c745-4998-b554-20170b805dca', 'type': 'tool_call'}]


In [60]:
def run_budget_agent(question, max_rounds=5):

    chat_history = []

    prompt_value = prompt.invoke({
        "chat_history": chat_history,
        "question": question
    })

    messages = list(prompt_value.messages)

    for round_no in range(max_rounds):

        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If the model does not request a tool,
        # we have reached the final answer.
        if not response.tool_calls:
            return response.content

        # Execute every requested tool call
        for tool_call in response.tool_calls:

            if tool_call["name"] == "search_budget":

                result = search_budget.invoke(
                    tool_call["args"]
                )

                messages.append(
                    ToolMessage(
                        content=result,
                        tool_call_id=tool_call["id"]
                    )
                )

    return "The answer is not available in the provided Budget document."

In [61]:
answer = run_budget_agent(
    "Who won the FIFA World Cup in 2018?"
)

print(answer)

The answer is not available in the provided Budget document.


## Q4 — Calculator Tool

In [62]:
import ast
import operator

@tool
def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression."""

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                evaluate(node.left),
                evaluate(node.right)
            )

        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                evaluate(node.operand)
            )

        raise ValueError("Unsupported arithmetic expression")

    try:
        tree = ast.parse(expression, mode="eval")
        return str(evaluate(tree.body))
    except Exception as e:
        return f"Calculation error: {e}"

print("Calculator tool created successfully.")

Calculator tool created successfully.


In [63]:
tools = [search_budget, calculator]

llm_with_tools_q4 = llm.bind_tools(tools)

print("Both search_budget and calculator tools are bound successfully.")

Both search_budget and calculator tools are bound successfully.


In [75]:
def run_q4_agent(question, max_rounds=8):

    chat_history = []

    # Reuse the SAME ChatPromptTemplate from Q3
    prompt_value = prompt.invoke({
        "chat_history": chat_history,
        "question": question
    })

    messages = list(prompt_value.messages)

    tool_map = {
        "search_budget": search_budget,
        "calculator": calculator
    }

    for round_no in range(max_rounds):

        print(f"\n--- ROUND {round_no + 1} ---")

        response = llm_with_tools_q4.invoke(messages)
        messages.append(response)

        # No tool call means the LLM has produced its final answer
        if not response.tool_calls:
            print("FINAL ANSWER:")
            print(response.content)
            return response.content

        # Execute all tools requested by the LLM
        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]

            print("Tool selected:", tool_name)
            print("Arguments:", tool_call["args"])

            if tool_name in tool_map:

                result = tool_map[tool_name].invoke(
                    tool_call["args"]
                )

                print("Tool result:", result)

                messages.append(
                    ToolMessage(
                        content=str(result),
                        tool_call_id=tool_call["id"]
                    )
                )

    return "Maximum number of rounds reached."

In [65]:
answer = run_q4_agent(
    "Calculate 125 * 24."
)


--- ROUND 1 ---
FINAL ANSWER:
125 × 24 = **3,000**.


In [66]:
answer = run_q4_agent(
    "Use the calculator tool to calculate 98765 * 4321."
)


--- ROUND 1 ---
Tool selected: calculator
Arguments: {'expression': '98765 * 4321'}
Tool result: 426763565

--- ROUND 2 ---
FINAL ANSWER:
The result of \(98765 \times 4321\) is **426,763,565**.


In [67]:
answer = run_q4_agent(
    "Find the Biopharma SHAKTI outlay from the Budget document, "
    "then calculate 10% of that amount. "
    "Use the available tools."
)

print("\nANSWER:")
print(answer)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'Biopharma SHAKTI outlay'}
Tool result: Page [8]: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet glob

## Q5 — Test Queries and Tool Decision Evaluation

In [68]:
q1 = "What is the outlay of Biopharma SHAKTI?"

answer_q1 = run_q4_agent(q1)

print("\nFINAL ANSWER:")
print(answer_q1)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'Biopharma SHAKTI outlay'}
Tool result: Page [8]: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet glob

In [69]:
q2 = "How does the 2026-27 fiscal deficit compare with 2025-26?"

answer_q2 = run_q4_agent(q2)

print("\nFINAL ANSWER:")
print(answer_q2)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': '2026-27 fiscal deficit 2025-26'}
Tool result: Page [22, 23]: 4.5 percent of GDP by 2025-26. In RE 2025-26, the fiscal deficit has been estimated at par with BE of 2025-26 at 4.4 percent of GDP. In line with the new fiscal prudence path of debt consolidation, the fiscal deficit in BE 2026-27 is estimated to be 4.3 percent of GDP.

Page [22, 23]: 4.5 percent of GDP by 2025-26. In RE 2025-26, the fiscal deficit has been estimated at par with BE of 2025-26 at 4.4 percent of GDP. In line with the new fiscal prudence path of debt consolidation, the fiscal deficit in BE 2026-27 is estimated to be 4.3 percent of GDP.

Page [22, 23]: 4.5 percent of GDP by 2025-26. In RE 2025-26, the fiscal deficit has been estimated at par with BE of 2025-26 at 4.4 percent of GDP. In line with the new fiscal prudence path of debt consolidation, the fiscal deficit in BE 2026-27 is estimated to be 4.3 percent of GDP.

--- ROUND 2 ---
FINAL ANSWER

In [70]:
q3 = "What changes for NCCD on chewing tobacco, and what is the effective duty?"

answer_q3 = run_q4_agent(q3)

print("\nFINAL ANSWER:")
print(answer_q3)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'NCCD chewing tobacco effective duty'}
Tool result: Page [65]: E. Central Excise Duty Changes
1., Description. = Chewing tobacco. 1., NCCD Rate.From (per cent) = 25%. 1., NCCD Rate.To (per cent) = 60%. 2., Description. = Jarda scented tobacco. 2., NCCD Rate.From (per cent) = 25%. 2., NCCD Rate.To (per cent) = 60%. 3., Description. = Other. 3., NCCD Rate.From (per cent) = 25%. 3., NCCD Rate.To (per cent) = 60%
Note: Description of entries is indicative. Notification/Tariff may be referred for complete description.
***effective duty rate will be maintained at 25% by notification

Page [65]: E. Central Excise Duty Changes
1., Description. = Chewing tobacco. 1., NCCD Rate.From (per cent) = 25%. 1., NCCD Rate.To (per cent) = 60%. 2., Description. = Jarda scented tobacco. 2., NCCD Rate.From (per cent) = 25%. 2., NCCD Rate.To (per cent) = 60%. 3., Description. = Other. 3., NCCD Rate.From (per cent) = 25%. 3., NCCD Rate.To (per

In [71]:
q4 = "Hi, how are you?"

answer_q4 = run_q4_agent(q4)

print("\nFINAL ANSWER:")
print(answer_q4)


--- ROUND 1 ---
FINAL ANSWER:
Hello! I'm just a bunch of code, but I'm here and ready to help you. How can I assist you today?

FINAL ANSWER:
Hello! I'm just a bunch of code, but I'm here and ready to help you. How can I assist you today?


In [72]:
q5 = "Explain fiscal deficit in simple words."

answer_q5 = run_q4_agent(q5)

print("\nFINAL ANSWER:")
print(answer_q5)


--- ROUND 1 ---
FINAL ANSWER:
Fiscal deficit is the amount by which a government's total spending exceeds its total revenue (excluding borrowing). In other words, it’s the gap that the government has to fill by borrowing money. Think of it like a household that spends more than it earns; it has to take out a loan to cover the difference. The fiscal deficit shows how much the government needs to borrow to finance its activities in a given year.

FINAL ANSWER:
Fiscal deficit is the amount by which a government's total spending exceeds its total revenue (excluding borrowing). In other words, it’s the gap that the government has to fill by borrowing money. Think of it like a household that spends more than it earns; it has to take out a loan to cover the difference. The fiscal deficit shows how much the government needs to borrow to finance its activities in a given year.


In [73]:
q6 = "What are the new income tax slab rates in this budget?"

answer_q6 = run_q4_agent(q6)

print("\nFINAL ANSWER:")
print(answer_q6)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'new income tax slab rates'}
Tool result: Page [46, 47]: (i) Reduction of rate of Minimum Alternate Tax (MAT) and allowance of set-off of brought forward MAT credit to companies shifting to the new tax regime
- To enable companies to shift to the new regime, MAT is proposed to be made as a final tax and the corresponding rate is reduced from 15% to 14%. There shall be no allowance of credit in future tax years in respect of such payment.
- Further, the set-off of any brought forward MAT credit available from prior to tax year 2026-27 will only be allowed to domestic companies which shift henceforth to the new regime.
- This set-off of MAT credit brought forward as on 1/4/2026 is proposed to be allowed in the new tax regime to domestic companies to the extent of 25% of their tax liability.
- The brought forward MAT credit shall be available only up to fifteenth year from the year when the corresponding credit was first a

In [76]:
q7 = "By what percentage does public capex rise from BE 2025-26 to FY2026-27?"

answer_q7 = run_q4_agent(q7)

print("\nFINAL ANSWER:")
print(answer_q7)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'public capex rise from BE 2025-26 to FY2026-27 percentage'}
Tool result: Page [23]: Revised Estimates 2025-26
95. The Revised Estimates of the non-debt receipts are ₹34 lakh crore of which the Centre's net tax receipts are ₹26.7 lakh crore. The Revised Estimate of the total expenditure is ₹49.6 lakh crore, of which the capital expenditure is about ₹11 lakh crore.

Page [23]: Revised Estimates 2025-26
95. The Revised Estimates of the non-debt receipts are ₹34 lakh crore of which the Centre's net tax receipts are ₹26.7 lakh crore. The Revised Estimate of the total expenditure is ₹49.6 lakh crore, of which the capital expenditure is about ₹11 lakh crore.

Page [23]: Revised Estimates 2025-26
95. The Revised Estimates of the non-debt receipts are ₹34 lakh crore of which the Centre's net tax receipts are ₹26.7 lakh crore. The Revised Estimate of the total expenditure is ₹49.6 lakh crore, of which the capital expenditure is 

In [77]:
q8 = """What is the combined outlay of Biopharma SHAKTI,
the Container Manufacturing scheme and CCUS?"""

answer_q8 = run_q4_agent(q8)

print("\nFINAL ANSWER:")
print(answer_q8)


--- ROUND 1 ---
Tool selected: search_budget
Arguments: {'query': 'Biopharma SHAKTI outlay Container Manufacturing scheme CCUS combined outlay'}
Tool result: Page [8]: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Cent

### Q5 — Interactive Chat Loop
The following loop allows the user to enter multiple queries. The agent decides whether to use the Budget search tool, calculator tool, both tools, or answer directly. Type `exit` to stop the conversation.

In [78]:
while True:
    user_query = input("You: ").strip()

    if user_query.lower() == "exit":
        print("Chat ended.")
        break

    answer = run_q4_agent(user_query)

    print("\nAssistant:")
    print(answer)
    print("-" * 70)

You: Hi, how are you?

--- ROUND 1 ---
FINAL ANSWER:
Hello! I'm just a bunch of code, but I'm here and ready to help you. How can I assist you today?

Assistant:
Hello! I'm just a bunch of code, but I'm here and ready to help you. How can I assist you today?
----------------------------------------------------------------------
You: exit
Chat ended.


In [79]:
import pandas as pd

evaluation_data = [
    {
        "Query": "1. What is the outlay of Biopharma SHAKTI?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "₹10,000 crore over 5 years (Page 8)"
    },
    {
        "Query": "2. How does the 2026-27 fiscal deficit compare with 2025-26?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "2025-26: 4.4% of GDP; 2026-27: 4.3% of GDP"
    },
    {
        "Query": "3. What changes for NCCD on chewing tobacco, and what is the effective duty?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "Effective duty maintained at 25% (Page 65)"
    },
    {
        "Query": "4. Hi, how are you?",
        "Searched": "N",
        "Tools Used": "None",
        "Final Answer": "Answered directly without document search"
    },
    {
        "Query": "5. Explain fiscal deficit in simple words.",
        "Searched": "N",
        "Tools Used": "None",
        "Final Answer": "Answered directly using general knowledge"
    },
    {
        "Query": "6. What are the new income tax slab rates in this budget?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "Retrieval unsuccessful; maximum rounds reached"
    },
    {
        "Query": "7. By what percentage does public capex rise from BE 2025-26 to FY2026-27?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "Approximately 8.9% (Pages 11-12)"
    },
    {
        "Query": "8. What is the combined outlay of Biopharma SHAKTI, Container Manufacturing and CCUS?",
        "Searched": "Y",
        "Tools Used": "search_budget",
        "Final Answer": "₹40,000 crore total (Pages 8, 9 and 12)"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,Query,Searched,Tools Used,Final Answer
0,1. What is the outlay of Biopharma SHAKTI?,Y,search_budget,"₹10,000 crore over 5 years (Page 8)"
1,2. How does the 2026-27 fiscal deficit compare...,Y,search_budget,2025-26: 4.4% of GDP; 2026-27: 4.3% of GDP
2,"3. What changes for NCCD on chewing tobacco, a...",Y,search_budget,Effective duty maintained at 25% (Page 65)
3,"4. Hi, how are you?",N,None,Answered directly without document search
4,5. Explain fiscal deficit in simple words.,N,None,Answered directly using general knowledge
5,6. What are the new income tax slab rates in t...,Y,search_budget,Retrieval unsuccessful; maximum rounds reached
6,7. By what percentage does public capex rise f...,Y,search_budget,Approximately 8.9% (Pages 11-12)
7,8. What is the combined outlay of Biopharma SH...,Y,search_budget,"₹40,000 crore total (Pages 8, 9 and 12)"


In [80]:
correct_decisions = 8
total_queries = 8

decision_accuracy = (correct_decisions / total_queries) * 100

print(f"Correct tool decisions: {correct_decisions}/{total_queries}")
print(f"Decision Accuracy: {decision_accuracy:.0f}%")

Correct tool decisions: 8/8
Decision Accuracy: 100%
